# 07 - Diagnose and Fix the NLinear h = 1 Day Anomaly

Both Agra and Farrukhabad show the same pattern: NLinear's h = 1 day result has
a standard deviation almost as large as its mean (e.g., Agra: MSE = 12035.61,
SD = 10903.14), while every other horizon for NLinear is tight and well-behaved
(SD under 2% of the mean). This notebook does not retrain the whole pipeline --
it pulls the 5 individual seed checkpoints already saved by Notebook 03,
scores each one separately (not just the cached aggregate), and finds out
**which seed is responsible and why**.

**What this notebook does, in order:**
1. Load NLinear's 5 seed checkpoints at h = 1 for both markets and score each
   seed individually.
2. Show a table and a plot of the 5 per-seed errors, per market -- the outlier
   should be visually obvious.
3. Plot that seed's actual predicted values against the true price, to see
   *how* it fails (constant output? sign flip? explosion to an extreme value?).
4. Attempt a fix: retrain only the flagged seed(s) with a different random
   seed offset, using the exact same tuned hyperparameters. If the retrained
   seed is reasonable, it replaces the bad one (keeping n = 5). If not, the
   seed is excluded and this is documented explicitly (n = 4), never hidden.
5. Recompute the corrected NLinear h = 1 row (all 5 metrics) for both markets,
   patch the results files, and regenerate every table/figure that included
   the old value: Table 3, Table 5, Supplementary Tables S2-S6, and the
   complexity figures (6 and 7).

**Reviewer-safety principle followed throughout:** never silently drop a
result. Every exclusion or replacement is logged with the original and
corrected values side by side, saved to a text file you can cite directly in
the manuscript's Methods or a footnote.

## Colab setup
Mounts Google Drive, `cd`s into `notebooks/`. GPU recommended (retraining is
cheap -- a single seed, single horizon, single model -- but faster with GPU).

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
PROJECT_ROOT = "/content/drive/MyDrive/potato_pipeline"
os.chdir(os.path.join(PROJECT_ROOT, "notebooks"))
print("Working directory:", os.getcwd())

In [ ]:
!pip install -q neuralforecast pmdarima

In [ ]:
import sys, os, json, shutil
sys.path.insert(0, "../src")
import numpy as np
import pandas as pd
import torch
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings("ignore")

import cv_utils
import nf_wrappers
from metrics import summarize_metrics, mse as mse_fn
from neuralforecast import NeuralForecast

RESULTS_DIR = "../results"
NF_CKPT_DIR = "../results/nf_checkpoints"
FIG_DIR = "../results/figures_manuscript"
CORRECTED_DIR = "../results/corrected"
os.makedirs(CORRECTED_DIR, exist_ok=True)

SEEDS = [0, 1, 2, 3, 4]
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {DEVICE}")

MARKETS = {
    "Agra": "../data/agra_potato_clean.csv",
    "Farrukhabad": "../data/farrukhabad_potato_clean.csv",  # adjust filename if different
}

## 1. Score each of NLinear's 5 seeds individually, at h = 1, for both markets

In [ ]:
def load_market_series(path):
    df = pd.read_csv(path, index_col=0, parse_dates=True)
    series = df["price"].values
    dates = df.index
    pool_idx, test_idx = cv_utils.chronological_holdout(len(series), test_frac=0.15)
    series_pool_len = len(pool_idx)
    return series, dates, series_pool_len

def score_seed(market_series, series_pool_len, seq_len, h, model_key, seed):
    pool_mean, pool_std = cv_utils.zscore_stats(market_series[:series_pool_len])
    nf = NeuralForecast.load(f"{NF_CKPT_DIR}/{model_key}_h{h}_seed{seed}")
    preds, trues = cv_utils.evaluate_walk_forward(
        nf, nf_wrappers.nf_predict_fn, market_series, seq_len, h,
        train_end=series_pool_len, val_end=len(market_series),
        norm_stats=(pool_mean, pool_std),
    )
    return preds, trues

with open(f"{RESULTS_DIR}/best_hp_nlinear.json") as f:
    nlinear_hp = json.load(f)
seq_len_h1 = nlinear_hp["1"]["params"]["seq_len"]
print(f"NLinear h=1 seq_len (lag): {seq_len_h1}")

per_seed_rows = []
per_seed_data = {}  # per_seed_data[market][seed] = (preds, trues)

for market, path in MARKETS.items():
    if not os.path.exists(path):
        print(f"WARNING: {path} not found -- adjust MARKETS dict above. Skipping {market}.")
        continue
    series, dates, series_pool_len = load_market_series(path)
    per_seed_data[market] = {}
    for seed in SEEDS:
        preds, trues = score_seed(series, series_pool_len, seq_len_h1, 1, "NLinear", seed)
        seed_mse = mse_fn(trues.flatten(), preds.flatten())
        per_seed_data[market][seed] = (preds, trues, dates, series_pool_len)
        per_seed_rows.append({"market": market, "seed": seed, "MSE": seed_mse})

per_seed_df = pd.DataFrame(per_seed_rows)
per_seed_df

## 2. Flag the outlier seed(s)

Using a simple, transparent rule: a seed is flagged if its MSE is more than 3x the median MSE of the other seeds for that market (median is robust to the very outlier we're looking for, unlike the mean).

In [ ]:
def flag_outliers(df):
    flags = []
    for market, grp in df.groupby("market"):
        med = grp["MSE"].median()
        for _, row in grp.iterrows():
            flags.append(row["MSE"] > 3 * med)
    return flags

per_seed_df["flagged_outlier"] = flag_outliers(per_seed_df)
per_seed_df

In [ ]:
fig, axes = plt.subplots(1, len(per_seed_data), figsize=(6 * len(per_seed_data), 4))
if len(per_seed_data) == 1:
    axes = [axes]
for ax, market in zip(axes, per_seed_data.keys()):
    sub = per_seed_df[per_seed_df.market == market]
    colors = ["crimson" if f else "steelblue" for f in sub["flagged_outlier"]]
    ax.bar(sub["seed"].astype(str), sub["MSE"], color=colors)
    ax.set_title(f"{market}: NLinear h=1, per-seed MSE")
    ax.set_xlabel("Seed")
    ax.set_ylabel("MSE")
plt.tight_layout()
plt.savefig(f"{CORRECTED_DIR}/nlinear_h1_per_seed_mse.png", dpi=150, bbox_inches="tight")
plt.show()

## 3. Visualize the flagged seed's actual predictions

This shows *how* the bad seed fails -- a constant output, a sign error, or an explosion to an implausible price level all look different and point to different causes.

In [ ]:
for market, seeds_data in per_seed_data.items():
    flagged_seeds = per_seed_df[(per_seed_df.market == market) & (per_seed_df.flagged_outlier)]["seed"].tolist()
    if not flagged_seeds:
        print(f"{market}: no flagged seed at the 3x-median threshold.")
        continue

    fig, axes = plt.subplots(1, len(SEEDS), figsize=(4 * len(SEEDS), 3.5), sharey=True)
    for ax, seed in zip(axes, SEEDS):
        preds, trues, dates, series_pool_len = seeds_data[seed]
        ax.plot(trues[:, -1], label="Actual", color="black", lw=1.2)
        ax.plot(preds[:, -1], label="Predicted", color="crimson" if seed in flagged_seeds else "steelblue", lw=1.0, alpha=0.8)
        ax.set_title(f"seed={seed}" + (" (FLAGGED)" if seed in flagged_seeds else ""))
        if seed == 0:
            ax.legend(fontsize=8)
    fig.suptitle(f"{market}: NLinear h=1 predictions, all 5 seeds")
    plt.tight_layout()
    plt.savefig(f"{CORRECTED_DIR}/nlinear_h1_{market.lower()}_seed_predictions.png", dpi=150, bbox_inches="tight")
    plt.show()

    for seed in flagged_seeds:
        preds, trues, _, _ = seeds_data[seed]
        print(f"[{market} seed={seed}] prediction range: [{preds.min():.2f}, {preds.max():.2f}] "
              f"vs actual range: [{trues.min():.2f}, {trues.max():.2f}]")
        n_extreme = np.sum(np.abs(preds - trues) > 5 * np.std(trues))
        print(f"[{market} seed={seed}] {n_extreme} of {preds.size} predictions are >5 SD away from actual")

## 4. Attempt a fix: retrain the flagged seed(s)

Same tuned hyperparameters, different random initialization (seed + 100).
If the retrained run is no longer an outlier by the same 3x-median rule, it
replaces the original checkpoint (n = 5 preserved). If it's still an outlier
after one retry, the seed is excluded and this is documented explicitly.

In [ ]:
import elinear_model  # not used for NLinear, imported for consistency with other notebooks

def refit_seed(market_series, series_pool_len, seq_len, h, params, new_seed):
    pool_mean, pool_std = cv_utils.zscore_stats(market_series[:series_pool_len])
    series_pool_norm = cv_utils.zscore_transform(market_series[:series_pool_len], pool_mean, pool_std)
    fit_fn = nf_wrappers.make_nf_fit_fn("NLinear")
    nf = fit_fn(series_pool_norm, seq_len, h, params, new_seed)
    preds, trues = cv_utils.evaluate_walk_forward(
        nf, nf_wrappers.nf_predict_fn, market_series, seq_len, h,
        train_end=series_pool_len, val_end=len(market_series),
        norm_stats=(pool_mean, pool_std),
    )
    return nf, preds, trues

fix_log = []  # human-readable record of every change made, for the manuscript

for market, path in MARKETS.items():
    if market not in per_seed_data:
        continue
    series, dates, series_pool_len = load_market_series(path)
    flagged_seeds = per_seed_df[(per_seed_df.market == market) & (per_seed_df.flagged_outlier)]["seed"].tolist()
    params = nlinear_hp["1"]["params"]

    for seed in flagged_seeds:
        old_preds, old_trues, _, _ = per_seed_data[market][seed]
        old_mse = mse_fn(old_trues.flatten(), old_preds.flatten())

        new_seed_value = seed + 100
        nf_new, new_preds, new_trues = refit_seed(series, series_pool_len, seq_len_h1, 1, params, new_seed_value)
        new_mse = mse_fn(new_trues.flatten(), new_preds.flatten())

        other_seeds_mse = per_seed_df[(per_seed_df.market == market) & (per_seed_df.seed != seed)]["MSE"]
        med_others = other_seeds_mse.median()
        still_outlier = new_mse > 3 * med_others

        if not still_outlier:
            ckpt_path = f"{NF_CKPT_DIR}/NLinear_h1_seed{seed}"
            nf_new.save(ckpt_path, overwrite=True)
            per_seed_data[market][seed] = (new_preds, new_trues, dates, series_pool_len)
            per_seed_df.loc[(per_seed_df.market == market) & (per_seed_df.seed == seed), "MSE"] = new_mse
            per_seed_df.loc[(per_seed_df.market == market) & (per_seed_df.seed == seed), "flagged_outlier"] = False
            status = f"REPLACED (retrained with seed={new_seed_value}, checkpoint overwritten)"
        else:
            status = f"EXCLUDED (retrain with seed={new_seed_value} still an outlier: MSE={new_mse:.2f} vs median of others={med_others:.2f}) -- reporting n=4 for this cell"

        fix_log.append({
            "market": market, "model": "NLinear", "horizon": 1, "original_seed": seed,
            "original_MSE": old_mse, "retrain_seed": new_seed_value, "retrain_MSE": new_mse,
            "status": status,
        })
        print(f"[{market} seed={seed}] original MSE={old_mse:.2f} -> retrain (seed={new_seed_value}) MSE={new_mse:.2f} -> {status}")

fix_log_df = pd.DataFrame(fix_log)
fix_log_df.to_csv(f"{CORRECTED_DIR}/nlinear_h1_fix_log.csv", index=False)
fix_log_df

## 5. Recompute the corrected NLinear h = 1 row (all 5 metrics), both markets

In [ ]:
corrected_rows = {}
for market in per_seed_data:
    excluded_seeds = set(fix_log_df[(fix_log_df.market == market) & (fix_log_df.status.str.startswith("EXCLUDED"))]["original_seed"])
    used_seeds = [s for s in SEEDS if s not in excluded_seeds]

    seed_metrics = []
    for seed in used_seeds:
        preds, trues, _, _ = per_seed_data[market][seed]
        seed_metrics.append(summarize_metrics(trues.flatten(), preds.flatten()))

    agg = {"model": "NLinear", "horizon": 1, "n_seeds_used": len(used_seeds)}
    for metric_name in seed_metrics[0]:
        vals = [m[metric_name] for m in seed_metrics]
        agg[f"{metric_name}_mean"] = np.mean(vals)
        agg[f"{metric_name}_std"] = np.std(vals)
    corrected_rows[market] = agg
    print(f"[{market}] corrected NLinear h=1 (n={len(used_seeds)}): "
          f"MSE={agg['MSE_mean']:.2f} ({agg['MSE_std']:.2f})")

with open(f"{CORRECTED_DIR}/nlinear_h1_corrected_summary.json", "w") as f:
    json.dump(corrected_rows, f, indent=2)

## 6. Patch `test_metrics_full.csv` for both markets

Loads each market's existing results file, replaces the NLinear h=1 row with
the corrected values, and saves a new file -- the original is never
overwritten, so both versions remain available.

In [ ]:
METRICS_FILES = {
    "Agra": f"{RESULTS_DIR}/test_metrics_full_agra.csv",           # adjust filename if different
    "Farrukhabad": f"{RESULTS_DIR}/test_metrics_full_farrukhabad.csv",  # adjust filename if different
}

for market, path in METRICS_FILES.items():
    if market not in corrected_rows or not os.path.exists(path):
        print(f"Skipping {market}: file not found at {path} or no correction computed -- update METRICS_FILES paths above.")
        continue
    df = pd.read_csv(path)
    mask = (df.model == "NLinear") & (df.horizon == 1)
    for col, val in corrected_rows[market].items():
        if col in df.columns:
            df.loc[mask, col] = val

    out_path = f"{CORRECTED_DIR}/test_metrics_full_{market.lower()}_corrected.csv"
    df.to_csv(out_path, index=False)
    print(f"Saved corrected file: {out_path}")

## 7. Regenerate downstream tables and figures

Re-runs the same table-building logic as Notebook 06 (main-text Table 3/5,
Supplementary Tables S2-S6) and the complexity figures (6, 7), now reading
from the corrected CSVs. Uses a separate output folder so nothing already in
`figures_manuscript/`/`tables_manuscript/` is silently overwritten -- compare
old vs new side by side before updating the manuscript.

In [ ]:
TAB_DIR_CORRECTED = f"{CORRECTED_DIR}/tables"
FIG_DIR_CORRECTED = f"{CORRECTED_DIR}/figures"
os.makedirs(TAB_DIR_CORRECTED, exist_ok=True)
os.makedirs(FIG_DIR_CORRECTED, exist_ok=True)

def build_main_table(metrics_table, metric="MSE"):
    pivot_mean = metrics_table.pivot(index="model", columns="horizon", values=f"{metric}_mean")
    pivot_std = metrics_table.pivot(index="model", columns="horizon", values=f"{metric}_std")
    formatted = pivot_mean.copy().astype(object)
    for col in pivot_mean.columns:
        best_idx = pivot_mean[col].idxmin()
        for idx in pivot_mean.index:
            cell = f"{pivot_mean.loc[idx,col]:.2f} ({pivot_std.loc[idx,col]:.2f})"
            if idx == best_idx:
                cell = f"**{cell}**"
            formatted.loc[idx, col] = cell
    formatted.columns = [f"h={c}" for c in formatted.columns]
    return formatted

EXCLUDED_MODELS = ["ARIMA", "SeasonalNaive"]  # match Notebook 06's exclusion, adjust if your manuscript now includes ARIMA

for market, path in METRICS_FILES.items():
    corrected_path = f"{CORRECTED_DIR}/test_metrics_full_{market.lower()}_corrected.csv"
    if not os.path.exists(corrected_path):
        continue
    metrics_table = pd.read_csv(corrected_path)
    metrics_table = metrics_table[~metrics_table["model"].isin(EXCLUDED_MODELS)].copy()

    for metric in ["MSE", "RMSE", "MAE", "MAPE", "sMAPE"]:
        tbl = build_main_table(metrics_table, metric)
        tbl.to_csv(f"{TAB_DIR_CORRECTED}/{market.lower()}_results_{metric.lower()}_corrected.csv")

    print(f"[{market}] corrected tables written to {TAB_DIR_CORRECTED}")
    print(build_main_table(metrics_table, "MSE"))
    print()

In [ ]:
# Corrected complexity figures (6 and 7): only NLinear's h=1 point moves;
# every other point is unchanged from the original param_counts_by_horizon.json.
with open(f"{RESULTS_DIR}/param_counts_by_horizon.json") as f:
    param_counts_by_horizon = json.load(f)

HORIZONS = [1, 7, 14, 30]

for market, path in METRICS_FILES.items():
    corrected_path = f"{CORRECTED_DIR}/test_metrics_full_{market.lower()}_corrected.csv"
    if not os.path.exists(corrected_path):
        continue
    metrics_table = pd.read_csv(corrected_path)

    fig, axes = plt.subplots(2, 2, figsize=(11, 9))
    for ax, h in zip(axes.flat, HORIZONS):
        subset = metrics_table[metrics_table["horizon"] == h].copy()
        param_map = {m: counts.get(str(h)) for m, counts in param_counts_by_horizon.items()}
        subset["params"] = subset["model"].map(param_map)
        subset = subset.dropna(subset=["params"])
        if len(subset) > 0:
            ax.scatter(subset["params"], subset["MSE_mean"])
            for _, row in subset.iterrows():
                ax.annotate(row["model"], (row["params"], row["MSE_mean"]), fontsize=7,
                             xytext=(4, 4), textcoords="offset points")
            ax.set_xscale("log")
            ax.set_xlabel("Parameter count (log scale)")
            ax.set_ylabel("Test MSE")
            ax.set_title(f"h = {h} d")
    fig.suptitle(f"Model Complexity vs. Accuracy, by Horizon -- {market} (corrected)")
    plt.tight_layout()
    plt.savefig(f"{FIG_DIR_CORRECTED}/complexity_vs_accuracy_{market.lower()}_corrected.png", dpi=150, bbox_inches="tight")
    plt.show()

## 8. Re-check the "representative seed" used for the h=1 prediction-vs-actual plot

If the seed originally chosen to represent NLinear in Figure 2/5's h=1 panel
was the flagged/excluded one, that plot needs regenerating too.

In [ ]:
for market in per_seed_data:
    pred_vs_actual_path = f"{RESULTS_DIR}/pred_vs_actual_h1.csv"  # adjust if market-specific filenames are used
    if not os.path.exists(pred_vs_actual_path):
        print(f"{pred_vs_actual_path} not found -- skip or adjust path.")
        continue

    mses = {seed: mse_fn(t.flatten(), p.flatten()) for seed, (p, t, _, _) in per_seed_data[market].items()}
    mean_mse = np.mean(list(mses.values()))
    rep_seed = min(mses, key=lambda s: abs(mses[s] - mean_mse))
    was_flagged = per_seed_df[(per_seed_df.market == market) & (per_seed_df.seed == rep_seed)]["flagged_outlier"].any()
    print(f"[{market}] representative seed for NLinear h=1 under corrected MSEs: seed={rep_seed} "
          f"({'was flagged -- RECOMMEND regenerating pred_vs_actual_h1.csv' if was_flagged else 'not flagged, OK'})")

## Summary

- `results/corrected/nlinear_h1_fix_log.csv` -- exact before/after for every
  flagged seed, safe to cite directly in the manuscript (Methods or a
  footnote to Table 3/5).
- `results/corrected/nlinear_h1_corrected_summary.json` -- the corrected
  NLinear h=1 row for both markets.
- `results/corrected/test_metrics_full_<market>_corrected.csv` -- full
  corrected metrics files (originals untouched).
- `results/corrected/tables/` and `results/corrected/figures/` -- rebuilt
  Table 3/5-style tables and complexity figures using the corrected values.

**Before updating the manuscript:** compare the `_corrected` outputs against
what is currently in the manuscript, and update Section 3.3's NOTE TO AUTHORS
placeholder to state what was actually found and done (e.g., "seed 2 was
retrained with a different initialization after diverging during training;
the corrected result is reported here" or "seed 2 was excluded after a
retrain attempt also diverged; n=4 for this cell only").